# Pet Healthcare — Kaggle CPU load baseline
Run on Kaggle **CPU** with internet enabled. Replace the public repository URL and exact commit below. Results remain under `/kaggle/working/nipopeto-evidence` for download and review.


In [ ]:
from pathlib import Path
import os, subprocess, shutil, hashlib, urllib.request
repo_url = "REPLACE_WITH_PUBLIC_GITHUB_URL"
commit = "REPLACE_WITH_FULL_COMMIT_SHA"
assert repo_url.startswith("https://github.com/") and len(commit) == 40
repo = Path("/kaggle/working/Pet-healthcare")
assert not repo.exists(), "Use a fresh Kaggle session or inspect the existing checkout before continuing."
subprocess.run(["git", "clone", repo_url, str(repo)], check=True)
subprocess.run(["git", "checkout", commit], cwd=repo, check=True)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())


In [ ]:
# Native PostgreSQL; no Docker daemon is required.
if not shutil.which("pg_config") or not shutil.which("createdb"):
    assert os.geteuid() == 0, "Install native PostgreSQL binaries in this Kaggle session."
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "postgresql", "postgresql-client", "libpq-dev"], check=True)
node_ok = False
if shutil.which("node"):
    major = int(subprocess.check_output(["node", "--version"], text=True).strip().lstrip("v").split(".")[0])
    node_ok = major == 24
if not node_ok:
    version = "v24.15.0"
    name = f"node-{version}-linux-x64.tar.xz"
    base = f"https://nodejs.org/dist/{version}/"
    archive = Path("/kaggle/working") / name
    urllib.request.urlretrieve(base + name, archive)
    checksums = urllib.request.urlopen(base + "SHASUMS256.txt").read().decode()
    expected = next(line.split()[0] for line in checksums.splitlines() if line.split()[-1] == name)
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Node download checksum mismatch"
    node_dir = Path("/kaggle/working/node24")
    node_dir.mkdir(exist_ok=False)
    subprocess.run(["tar", "-xJf", str(archive), "-C", str(node_dir), "--strip-components=1"], check=True)
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
subprocess.run(["python", "-m", "pip", "install", "-r", str(repo / "benchmarks/kaggle/requirements.txt")], check=True)
print("Node:", subprocess.check_output(["node", "--version"], text=True).strip())
print("PostgreSQL:", subprocess.check_output(["pg_config", "--version"], text=True).strip())
print("Locust:", subprocess.check_output(["locust", "--version"], text=True).strip())


In [ ]:
# Fixed matrix: two profiles, 1/10/50/100 users, two repeats. Stop on declared threshold.
evidence = Path("/kaggle/working/nipopeto-evidence")
label = "B1"  # Phase 1 compliant baseline; use B2 for a later improvement in the same CPU session.
evidence.mkdir(exist_ok=True)
for profile in ("ReadHeavyUser", "MixedUser"):
    for users in (1, 10, 50, 100):
        for repeat in (1, 2):
            folder = evidence / f"{label}_{profile}_{users}_{repeat}"
            args = ["python", str(repo / "benchmarks/kaggle/runner.py"), "--repo", str(repo), "--output", str(folder), "--profile", profile, "--users", str(users), "--spawn-rate", "5", "--warmup", "15s", "--duration", "60s", "--seed", "phase1"]
            print("Running", folder.name, flush=True)
            result = subprocess.run(args, cwd=repo, env=os.environ.copy())
            if result.returncode:
                raise RuntimeError(f"Run {folder.name} stopped with code {result.returncode}; inspect manifest/logs before escalating load")
print("Evidence saved in", evidence)
